# Advanced Problems with Solutions: Decimal Constructors and Contexts

Focus: `Decimal` construction from integers, strings, tuples, floats, context precision, local contexts, and best practices.

In [1]:
import decimal
from decimal import Decimal

## Problem 1: Constructor Precision vs Context Precision

Set the global decimal precision to `3`. Then create this Decimal:

```python
x = Decimal('12345.6789')
```

Show that the constructor preserves all digits, but arithmetic operations use the active context.

In [2]:
# Solution

decimal.getcontext().prec = 3

x = Decimal('12345.6789')

print('Constructed Decimal:', x)
print('After arithmetic:', x + Decimal('1'))

Constructed Decimal: 12345.6789
After arithmetic: 1.23E+4


The constructor does not round based on context precision. However, arithmetic operations such as addition, multiplication, and division do use the active context.

## Problem 2: Why `Decimal(float)` Is Dangerous

Compare the following:

```python
Decimal(0.1)
Decimal('0.1')
```

Then explain why the first value is much longer.

In [3]:
# Solution

from_float = Decimal(0.1)
from_string = Decimal('0.1')

print('Decimal(0.1):')
print(from_float)

print('\nDecimal("0.1"):')
print(from_string)

print('\nAre they equal?', from_float == from_string)

Decimal(0.1):
0.1000000000000000055511151231257827021181583404541015625

Decimal("0.1"):
0.1

Are they equal? False


`0.1` as a Python float is already a binary approximation before it reaches the `Decimal` constructor. `Decimal(0.1)` converts that binary approximation exactly. Use strings for exact decimal values.

## Problem 3: Tuple-Based Decimal Construction

Construct the following numbers using tuple syntax:

- `-42.75`
- `0.00314`
- `9.99E+4`

Recall that the tuple form is:

```python
Decimal((sign, digits, exponent))
```

In [4]:
# Solution

a = Decimal((1, (4, 2, 7, 5), -2))
b = Decimal((0, (3, 1, 4), -5))
c = Decimal((0, (9, 9, 9), 2))

print(a)
print(b)
print(c)

-42.75
0.00314
9.99E+4


The tuple constructor is exact and bypasses binary floating-point issues. The exponent controls where the decimal point is placed.

## Problem 4: Local Context Does Not Rewrite Existing Decimals

Create two high-precision Decimals outside a local context. Then enter a local context with precision `4` and add them. Show that only the result of the operation is rounded, not the original operands.

In [5]:
# Solution

a = Decimal('1.23456789')
b = Decimal('9.87654321')

with decimal.localcontext() as ctx:
    ctx.prec = 4
    c = a + b
    print('a inside context:', a)
    print('b inside context:', b)
    print('c inside context:', c)

print('a after context:', a)
print('b after context:', b)
print('c after context:', c)

a inside context: 1.23456789
b inside context: 9.87654321
c inside context: 11.11
a after context: 1.23456789
b after context: 9.87654321
c after context: 11.11


Contexts affect operations, not the stored digits of existing `Decimal` objects.

## Problem 5: Significant Digits, Not Decimal Places

Use precision `3` and compute:

```python
Decimal('999') + Decimal('1')
Decimal('0.00999') + Decimal('0.00001')
```

Explain why both results are governed by significant digits.

In [6]:
# Solution

with decimal.localcontext() as ctx:
    ctx.prec = 3
    x = Decimal('999') + Decimal('1')
    y = Decimal('0.00999') + Decimal('0.00001')

print(x)
print(y)

1.00E+3
0.0100


Decimal precision means total significant digits, not digits after the decimal point.

## Problem 6: Building a Safe Decimal Constructor

Write a function `make_decimal(value)` that accepts integers and strings, but rejects floats with a clear error message.

In [7]:
# Solution

def make_decimal(value):
    if isinstance(value, float):
        raise TypeError('Do not construct Decimal from float. Use a string instead.')
    return Decimal(value)


print(make_decimal(10))
print(make_decimal('0.1'))

try:
    print(make_decimal(0.1))
except TypeError as ex:
    print(type(ex).__name__, ex)

10
0.1
TypeError Do not construct Decimal from float. Use a string instead.


Rejecting floats at construction time prevents silent precision bugs from entering the system.

## Problem 7: Context-Dependent Result Persistence

Inside a local context with precision `2`, compute a value `c = a + b`. After leaving the context, set the global precision to `20`. Show that `c` does not regain lost precision.

In [8]:
# Solution

a = Decimal('0.12345')
b = Decimal('0.12345')

with decimal.localcontext() as ctx:
    ctx.prec = 2
    c = a + b

decimal.getcontext().prec = 20

print('a:', a)
print('b:', b)
print('c:', c)
print('Recomputed with higher precision:', a + b)

a: 0.12345
b: 0.12345
c: 0.25
Recomputed with higher precision: 0.24690


Once an operation has produced a rounded result, the discarded digits are gone. To get a more precise value, recompute from the original operands under a higher precision context.

## Problem 8: Audit Float Contamination in a Dataset

Given mixed input values, separate safe values from unsafe float values before constructing Decimals.

```python
values = ['10.25', 3, '0.1', 0.2, Decimal('4.50'), 7.75]
```

In [9]:
# Solution

values = ['10.25', 3, '0.1', 0.2, Decimal('4.50'), 7.75]

safe_decimals = []
unsafe_values = []

for value in values:
    if isinstance(value, float):
        unsafe_values.append(value)
    else:
        safe_decimals.append(Decimal(value))

print('Safe Decimals:', safe_decimals)
print('Unsafe float inputs:', unsafe_values)

Safe Decimals: [Decimal('10.25'), Decimal('3'), Decimal('0.1'), Decimal('4.50')]
Unsafe float inputs: [0.2, 7.75]


In production-quality decimal workflows, float inputs should usually be rejected, converted upstream to strings, or handled with explicit awareness of the precision loss.

## Problem 9: Comparing Constructor and Operation Rounding

Set precision to `2`. Create `Decimal('1.2345')`. Then compute:

```python
x
x + Decimal('0')
x * Decimal('1')
```

Explain why the first expression differs from the arithmetic expressions.

In [10]:
# Solution

with decimal.localcontext() as ctx:
    ctx.prec = 2
    x = Decimal('1.2345')

    print('x:', x)
    print('x + 0:', x + Decimal('0'))
    print('x * 1:', x * Decimal('1'))

x: 1.2345
x + 0: 1.2
x * 1: 1.2


Constructing `x` preserves the input digits. Arithmetic operations apply the active context and may round the result.

## Problem 10: Robust Decimal Summation

Write a function `decimal_sum(values, precision)` that:

- rejects floats,
- constructs Decimals safely,
- sums under a local context,
- does not mutate the global context.

In [11]:
# Solution

def decimal_sum(values, precision):
    decimals = []

    for value in values:
        if isinstance(value, float):
            raise TypeError(f'Float input rejected: {value!r}')
        decimals.append(Decimal(value))

    with decimal.localcontext() as ctx:
        ctx.prec = precision
        return sum(decimals, Decimal('0'))


before = decimal.getcontext().prec

result = decimal_sum(['0.12345', '0.12345', '1000'], precision=5)

after = decimal.getcontext().prec

print('Result:', result)
print('Global precision before:', before)
print('Global precision after:', after)

Result: 1000.2
Global precision before: 20
Global precision after: 20


This function follows best practices: exact construction, float rejection, local context isolation, and explicit precision control.

## Best Practices Summary

- Use `Decimal('...')` for exact decimal literals.
- Avoid `Decimal(float)` unless you intentionally want the exact decimal value of the binary float.
- Context precision affects arithmetic operations, not Decimal construction.
- Local contexts are safer than mutating the global context.
- Results created under low precision remain low precision after the context exits.
- Precision means significant digits, not decimal places.
- Reject or audit float inputs in financial and precision-sensitive systems.